In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 46. Week 32 — MCMC, ESS, split-Rhat, and approximation boundaries

## 学習目標

- random-walk Metropolisのdetailed balanceを説明できる
- acceptance rate、ESS、multi-chain split-(\hat R)を併用できる
- proposal scale failureを実験で示せる
- Gibbs、HMC/NUTS、VI、SMCの適用条件と本Coreの境界を説明できる

## 前提知識

- Week 29のposterior
- B2のMarkov chainとMonte Carlo error

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 46


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Random-walk Metropolis

対称proposal (q(\theta'\mid\theta)=q(\theta\mid\theta')) ならacceptanceは

$$
\alpha(\theta,\theta')=\min\left\{1,\frac{p(\theta'\mid y)}{p(\theta\mid y)}\right\}.
$$

高いacceptanceだけでは良いmixingを意味しない。小さすぎるstepはほぼ全てacceptされてもESSが低い。

In [4]:
horizon = 5
origins = np.arange(curve_yields.shape[0] - horizon)
target = (curve_yields[origins + horizon, 3] - curve_yields[origins, 3]) * 100.0
target_dates = curve_dates[origins + horizon]
training_target = target[target_dates <= train_end_date]
known_variance = float(np.var(training_target, ddof=1))
prior_variance = 25.0

def log_posterior(parameter):
    mean = parameter[0]
    return float(
        -0.5 * mean**2 / prior_variance
        -0.5 * np.sum((training_target - mean) ** 2) / known_variance
    )

chains = []
chain_rows = []
for chain_index, initial in enumerate([-5.0, -1.0, 1.0, 5.0]):
    result = qt.metropolis_hastings(
        log_posterior,
        [initial],
        3000,
        proposal_scale=[0.25],
        burn_in=1000,
        rng=task_rng(1, chain_index),
    )
    chains.append(result.samples[:, 0])
    chain_rows.append(
        {"chain": chain_index, "acceptance_rate": result.acceptance_rate, "ess": result.effective_sample_size[0], "mean": result.samples[:, 0].mean()}
    )
chain_array = np.asarray(chains)[:, :, None]
display(pd.DataFrame(chain_rows))
print("classical split-Rhat:", qt.split_rhat(chain_array)[0])

,chain,acceptance_rate,ess,mean
0,0,0.68450,406.728143,-0.209322
1,1,0.68075,326.613410,-0.224778
2,2,0.70075,349.710404,-0.226632
3,3,0.69525,470.711558,-0.227594


classical split-Rhat: 1.0030047648713492


In [5]:
fig = go.Figure()
for chain_index in range(chain_array.shape[0]):
    fig.add_scatter(x=np.arange(500), y=chain_array[chain_index, :500, 0], name=f"chain {chain_index}", mode="lines")
fig.update_layout(title="Multiple-chain trace audit", xaxis_title="Retained draw", yaxis_title="Mean change (bp)", template="plotly_white")
fig.show()

## 2. Proposal-scale failure

In [6]:
proposal_rows = []
for index, proposal_scale in enumerate([0.002, 0.25, 10.0]):
    result = qt.metropolis_hastings(
        log_posterior,
        [0.0],
        3000,
        proposal_scale=[proposal_scale],
        burn_in=500,
        rng=task_rng(2, index),
    )
    proposal_rows.append(
        {"proposal_scale": proposal_scale, "acceptance_rate": result.acceptance_rate, "ess": result.effective_sample_size[0], "posterior_mean": result.samples[:, 0].mean()}
    )
display(pd.DataFrame(proposal_rows))

,proposal_scale,acceptance_rate,ess,posterior_mean
0,0.002,0.997429,2.518377,-0.019340
1,0.250,0.688000,332.917804,-0.198683
2,10.000,0.032571,67.516131,-0.208271


## 3. Algorithm boundary

| Method | Strength | Required audit |
|---|---|---|
| Gibbs | conjugate full conditionals | autocorrelation、blocking |
| MH | generic ratio | proposal scale、ESS、multi-chain |
| HMC/NUTS | continuous differentiable high dimension | divergences、energy、rank-normalized (\hat R)/ESS |
| VI | fast approximation | under-dispersion、objective gap、predictive calibration |
| SMC | sequence/multimodality | particle ESS、resampling、path degeneracy |

本Core helperのsplit-(\hat R)はclassical variance versionであり、Vehtari et al.のrank-normalized/folded/localized diagnosticを実装していない。production HMC/NUTSの代替としない。

## 4. 失敗モード

- 一つのchainとtrace plotだけでconvergenceを宣言する
- acceptance rateだけを最適化する
- burn-in後もinitialization差が残るのに平均を統合する
- classical split-(\hat R)をrank-normalized (\hat R)と呼ぶ
- finite differenceをautomatic differentiationと呼ぶ

## 5. 段階別演習

### 基礎

1. three proposal scalesのacceptance/ESS trade-offを説明せよ。
2. Monte Carlo SEをposterior SDとESSから計算せよ。

### 標準

3. rank normalizationを実装しclassical split-(\hat R)と比較せよ。
4. independent conjugate drawsをoracleとしてMH meanを検証せよ。

### 研究

5. HMC/NUTS導入時のdependency、AD、divergence test計画を書け。

## 6. Exit Criteria

- [ ] explicit RNGと複数chainを使った
- [ ] acceptance、ESS、split-(\hat R)を報告した
- [ ] proposal-scale failureを再現した
- [ ] classicalとrank-normalized diagnosticを区別した
- [ ] HMC/NUTS/VI/SMCを未実装のAdvanced範囲と明記した

## 7. 出典


- [Gelman et al., Bayesian Data Analysis, 3rd ed.](https://sites.stat.columbia.edu/gelman/book/)
- [Gelman et al., Bayesian Workflow](https://arxiv.org/abs/2011.01808)
- [Vehtari, Gelman, and Gabry, Practical Bayesian model evaluation](https://doi.org/10.1007/s11222-016-9696-4)

- [Rabiner (1989), A Tutorial on Hidden Markov Models](https://www.cs.cmu.edu/~durand/03-711/Readings/Rabiner89.pdf)
- [Vehtari et al., Rank-normalization, folding, and localization](https://arxiv.org/abs/1903.08008)
- [Stan Reference Manual — MCMC Sampling](https://mc-stan.org/docs/reference-manual/mcmc.html)